# Classification de sudokus complétés : valide ou invalide

> ⚠️ *Ce notebook est **long à exécuter**. C'est une démonstration que vous devriez laisser tourner en arrière-plan pendant qu'on regarde un autre module.*

Pendant qu'on continue avec le prochain module, on va lancer l'entraînement de différents algorithmes sur un nouveau problème : **apprendre à distinguer des grilles de sudoku valides et invalides**.

Avez un peu de temps, vous seriez capables d'écrire une fonction qui vérifie qu'un sudoku est valide ou non (ie, qu'il comporte seulement des chiffres de 1 à 9, pas de chiffres répétés sur les lignes, les colonnes et les blocs 3x3).

&nbsp;

On va plutôt essayer de laisser des modèles d'apprentissage se débrouiller tout seuls : on va utiliser un grand nombre de grilles pré-identifiées comme valides ou non, et on va fait des `.fit()` sur différents modèles pour comparer les performances.

In [1]:
#@title Importations
import numpy as np
import pandas as pd
import plotly.express as px

# Modèles
from sklearn.svm import LinearSVC, SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

# Métriques pour évaluer le résultat
from sklearn.metrics import accuracy_score, confusion_matrix

In [2]:
#@title Jeu de données: 800 000 grilles de sudokus étiquettées comme valides/invalides
! (test -e sudokus.pkl && echo "Fichier prêt") || ( wget https://raw.githubusercontent.com/316k/misc-data/refs/heads/main/sudokus.pkl.gz -O sudokus.pkl.gz && gunzip sudokus.pkl.gz )

import pickle
with open('sudokus.pkl', 'rb') as f:
  dataset = pickle.load(f)

--2026-05-04 11:46:48--  https://raw.githubusercontent.com/316k/misc-data/refs/heads/main/sudokus.pkl.gz
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 47740340 (46M) [application/octet-stream]
Saving to: ‘sudokus.pkl.gz’

sudokus.pkl.gz      100%[===================>]  45.53M   102MB/s    in 0.4s    

2026-05-04 11:46:50 (102 MB/s) - ‘sudokus.pkl.gz’ saved [47740340/47740340]



In [3]:
#@title Visualisation du jeu de données

def afficher_sudoku(s):
  grille = s.reshape(9, 9)
  for i in range(9):
    if i in [3, 6]:
      print("  ", '-' * (9+2))
    print("   ", end="")
    for j in range(9):
      print(int(grille[i, j]), end='')
      if j in [2, 5]:
        print('|', end='')
    print()


def afficher_grille(i=0):
  print(f"Grille #{i}:", "Sudoku valide" if dataset[i, -1] else "Sudoku invalide")
  print()
  afficher_sudoku(dataset[i, :-1])
  print()
  print("X[i] = ", dataset[i, :-1])
  print("y[i] = ", dataset[i, -1])

from ipywidgets import interact
out = interact(afficher_grille, i=(0, len(dataset)-1))

interactive(children=(IntSlider(value=0, description='i', max=799999), Output()), _dom_classes=('widget-intera…

## Séparation du jeu de données

Comme d'habitude, on va séparer nos données en trois, car on va comparer des modèles différents avec différentes configurations d'hyperparamètres.

In [4]:
total_size = len(dataset)
train_size = int(0.6*total_size)
valid_size = int(0.2*total_size)
test_size = total_size - (train_size + valid_size)

data_entrainement = dataset[:train_size]
data_validation = dataset[train_size:train_size + valid_size]
data_test = dataset[train_size + valid_size:]

X_entrainement = data_entrainement[:, :-1]
y_entrainement = data_entrainement[:, -1]

X_validation = data_validation[:, :-1]
y_validation = data_validation[:, -1]

X_test = data_test[:, :-1]
y_test = data_test[:, -1]

## Mise à l'échelle des données

Pour rappel : plusieurs algorithmes d'apprentissage ont **absolument** besoin que les données soient mises sur une même échelle, pour deux grandes raisons :

1. Dans certains algorithmes (ex.: KNN et SVM), la notion de distance entre les vecteurs est utilisée (donc des dimensions sur des échelles différentes vont biaiser l'algo dans la direction des plus grosses valeurs)
2. La descente de gradient, et plus généralement, beaucoup d'algorithmes d'optimisation numérique, se comportent beaucoup mieux sur des données mises à l'échelle. **Ça aide à converger vite**.

In [5]:
#@title Mise à l'échelle des données avec un outil de sklearn
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaler.fit(X_entrainement)

X_entrainement = scaler.transform(X_entrainement)
X_validation = scaler.transform(X_validation)
X_test = scaler.transform(X_test)

## Tester un SVM sur notre problème

Avant de tester des réseaux de neurones, on devrait commencer par tester un algorithme plus simple.

In [6]:
#@title Entraînement de SVM sur différents hyperparamètres

# Variable qui va contenir les résultats de nos tests
resultats = [
    # description du modèle, exactitude d'entraînement, exactitude de test
]

for C in 10.0**np.arange(-4, 2): # 0.00001, 0.0001, ... jusqu'à 10
  modele = LinearSVC(C=C)

  modele.fit(X_entrainement, y_entrainement)

  y_predictions_entrainement = modele.predict(X_entrainement)
  y_predictions_validation = modele.predict(X_validation)

  stats = [
    f"SVM avec C={C}",
    accuracy_score(y_entrainement, y_predictions_entrainement),
    accuracy_score(y_validation, y_predictions_validation),
  ]

  print(*stats)
  resultats.append(stats)

pd.DataFrame(resultats, columns=["Description", "Exactitude d'entraînement", "Exactitude de validation"])

SVM avec C=0.0001 0.5150541666666667 0.5088625
SVM avec C=0.001 0.5608020833333334 0.55620625
SVM avec C=0.01 0.6581479166666667 0.6603
SVM avec C=0.1 0.6516083333333333 0.65464375
SVM avec C=1.0 0.6515291666666667 0.65458125
SVM avec C=10.0 0.6515291666666667 0.65458125


,Description,Exactitude d'entraînement,Exactitude de validation
0,SVM avec C=0.0001,0.515054,0.508862
1,SVM avec C=0.001,0.560802,0.556206
2,SVM avec C=0.01,0.658148,0.660300
3,SVM avec C=0.1,0.651608,0.654644
4,SVM avec C=1.0,0.651529,0.654581
5,SVM avec C=10.0,0.651529,0.654581


**Exercice**

A) Tentez d'expliquer ce qui va se passer lorsqu'on donne une nouvelle grille de sudoku à classifier à un SVM

B) Croyez-vous que ce modèle va donner des bons résultats pour le problème de valider si un sudoku est bon ou non?

**RÉPONSE**

A) SVM va apprendre une somme pondérée des cases de la grille de sudoku, de la forme :

$$m_1 x_1 + m_2 x_2 + m_3 x_3 + \ldots + m_{81} x_{81} + b = 0$$

Pour dire si un sudoku est valide ou non, il va donc regarder :

$$m_1 x_1 + m_2 x_2 + m_3 x_3 + \ldots + m_{81} x_{81} < -b$$

Comme il n'y a pas de case plus importante que les autres, les coefficients deveraient avoir à peu près les mêmes valeurs.

&nbsp;

Le résultat du SVM ne devrait pas être génial, mais on a des chances d'avoir un modèle *un peu mieux que le hasard*.

Au mieux, il pourrait regarder si la somme de toutes les cases est en dessous de $9*1 + 9*2 + 9*3 + ... 9*9$, donc arriver à détecter certains cas bizarres (ex.: il y a trop de 1 dans la grille, la somme de toutes les cases est plus petite que celle d'un sudoku valide).

# Réseau de neurones

Avec la bonne configuration, un réseau de neurones peut apprendre n'importe quoi.

En théorie, on sait que c'est possible d'apprendre n'importe quelle fonction, il reste la question de réussir à trouver des hyperparamètres qui nous permettent d'arriver à la bonne réponse.

On doit choisir correctement :

- La configuration des couches cachées
- Le taux d'apprentissage
- Le taux de régularisation
- La fonction d'activation (ReLU ou tanh)

Ces quatre-là sont le strict minimum, mais il y en a d'autres qui sont utiles pour des cas plus avancés.

On pourrait tenter d'ajuster la *tolérance* de la descente de gradient par exemple, ou encore des hyperparamètres requis quand on utilise une variation de la descente de gradient. On peut par exemple configurer un genre d'*inertie* pour la descente de gradient, ou encore modifier graduellement le taux d'apprentissage avec le temps. (Lisez sur l'*optimiseur ADAM* si ces possibilités vous intéressent, c'est celui qui est utilisé par défaut par `sklearn`)

&nbsp;

**Il y a trop de combinaisons d'hyperparamètres pour arriver à tout tester.**

Plutôt : chaque personne dans la classe va lancer quelques entraînements **au hasard**, et on mettra les résultats en commun à la fin du cours.

In [7]:
#@title Définir l'espace des hyperparamètres

choix_couches_cachees = [
    (5_000,),
    (800, 512, 32),
    (300, 600, 1000),
    (500, 500),
    (500, 250, 16),
    (400,),
    (15, 15, 15, 15, 15, 15, 15, 15, 15),
    (100, 100, 100, 100, 100, 100, 100),
    (50, 150, 300, 500, 300, 150, 50),
    (20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20, 20),
]

# Choix possibles pour la fonction d'activation
choix_fonction_activation = ['relu', 'tanh']

# Valeurs possibles pour le taux d'apprentissage
choix_taux_apprentissage = 10.0**np.arange(-5, 0, 1)

# Choix possibles pour le taux de régularisation (alpha)
choix_taux_regularisation = 10.0**np.arange(-5, 2, 1)

#########

nb_combinaisons_choix_possibles = (
      len(choix_taux_apprentissage)
    * len(choix_fonction_activation)
    * len(choix_taux_regularisation)
    * len(choix_couches_cachees)
)

print("Nombre de configurations d'hyperparamètres possibles:", nb_combinaisons_choix_possibles)

Nombre de configurations d'hyperparamètres possibles: 700


On va utiliser `partial_fit()` pour avoir la qualité de chaque modèle entraîné après chaque époque.

In [ ]:
#@title Entraînement de réseaux de neurones avec des hyperparamètres au hasard
from random import choice
from time import time

resultats_reseaux_neurones = []

for i in range(10):
  couches_cachees = choice(choix_couches_cachees)
  fonction_activation = choice(choix_fonction_activation)
  taux_apprentissage = choice(choix_taux_apprentissage)
  taux_regularisation = choice(choix_taux_regularisation)

  description = f"Réseau de neurones {couches_cachees}, {fonction_activation}, taux={taux_apprentissage}, alpha={taux_regularisation}"

  print("===", description, "===")

  modele = MLPClassifier(
    couches_cachees,
    fonction_activation,
    learning_rate_init=taux_apprentissage,
    alpha=taux_regularisation,
  )

  modele.partial_fit(X_entrainement, y_entrainement, [0.0, 1.0])

  # Temps limite pour entraîner chaque modèle
  temps_debut = time()
  secondes_max = 60 * 10 # 60s * 10 = 10 minutes d'entraînement par modèle

  epoque = 0
  while time() - temps_debut < secondes_max:
    epoque += 1

    temps_debut_epoque = time()

    print(f"Époque #{epoque}")
    modele.partial_fit(X_entrainement, y_entrainement)
    print("Valeur de la fonction de perte minimisée:", modele.loss_)

    y_predictions_validation = modele.predict(X_validation)
    print("Exactitude de validation", accuracy_score(y_predictions_validation, y_validation))
    print("Temps requis pour l'époque:", round(time() - temps_debut_epoque, 3), "s")
    print()

  display(px.line(modele.loss_curve_, title="Visualisation de la perte minimisée selon l'époque d'entraînement"))

  y_predictions_entrainement = modele.predict(X_entrainement)
  y_predictions_validation = modele.predict(X_validation)

  stats = [
    description,
    accuracy_score(y_entrainement, y_predictions_entrainement),
    accuracy_score(y_validation, y_predictions_validation),
  ]

  resultats_reseaux_neurones.append(stats)

pd.DataFrame(resultats_reseaux_neurones, columns=["Description", "Exactitude d'entraînement", "Exactitude de validation"])

## Recherche plus fine

Une fois la recherche au hasard faite, on devrait avoir une idée de quelles valeurs d'hyperparamètres ont l'air de donner les meilleurs résultats.

On peut alors recommencer la recherche au hasard en repartant des quelques valeurs d'hyperparamètres qui semblaient bien marcher, et chercher des valeurs plus précises.

&nbsp;

Une fois qu'on a un modèle satisfaisant, on peut utiliser le jeu de données de test pour avoir une estimation qui n'est pas trop optimiste de l'erreur que notre modèle va faire sur des nouvelles données.

## Réduire le temps d'entraînement

Vous devriez vous rendre compte ici que l'entraînement d'un réseau de neurones est **très, très lent**.

On a à peine le temps dans un seul cours de tester une dizaine de combinaisons d'hyperparamètres...

&nbsp;

### CPU vs GPU

Un ordinateur moderne a généralement deux unités de calcul :

- Le CPU, ou l'unité centrale
- Le GPU, ou le *processeur graphique*

Quand vous écrivez du code python, de base, ça s'exécute sur le CPU. Le CPU est fait pour exécuter une seule instruction à la fois.

Le GPU, par contraste, est fait pour exécuter plusieurs instructions en même temps (possiblement plusieurs milliers d'instructions à la fois).

&nbsp;

> Pourquoi est-ce qu'on n'utilise pas le GPU en python dans ce cas?

Considérez le code suivant :
```python
x = 10
x *= 2
x += 5
print(x)
```

Essayer d'exécuter les lignes de ce code en même temps n'aurait aucun sens. Chaque ligne dépend du résultat de la précédente.

**Si on veut pouvoir utiliser le GPU dans notre programme, le code doit avoir été pensé pour ça**. De base, le code python que vous avez écrit dans votre premier cours de programmation n'est pas pensé pour ça. Le code fourni par `sklearn` n'est également pas pensé pour ça.

&nbsp;

En revanche, d'autres bibliothèques de fonctions le sont. En particulier : **`pytorch`** est une bibliothèque populaire en python pour définir et entraîner des réseaux de neurones, et la majorité de ses fonctions peuvent s'exécuter sur un GPU.

&nbsp;

À noter : un GPU consomme plus d'énergie qu'un CPU (de l'ordre de ≈2x), et Google Colab limite l'utilisation de ses GPUs pour les comptes gratuits.

Si vous souhaitez expérimenter avec pytorch, la version gratuite de Google Colab n'est pas le meilleur environnement.

### Modèles pré-entraînés

Depuis le début de la session, on entraîne des modèles de zéro, mais **une fois qu'on a trouvé les paramètres de notre modèle, on peut les enregistrer dans un fichier quelque part pour ne pas avoir à recommencer le processus à chaque fois**.

Il n'est pas rare pour des réseaux de neurones qui font des tâches complexes d'avoir plusieurs millions/milliards de paramètres, et l'entraînement peut durer des mois et des mois.

&nbsp;

La bibliothèque standard **`pickle`** de python permet d'enregistrer dans un fichier à peu près n'importe quel objet python, incluant tous les modèles de `sklearn` que vous pourriez vouloir entraîner. Pour sauvegarder un objet, on peut utiliser :

In [8]:
import pickle

objet = [1, 2, 3, "ceci est un objet écrit dans un fichier"]

with open("test-pickle.pkl", "wb") as f: # wb = write + binaire
  pickle.dump(objet, f)

# => Vous trouverez "objet.pkl" sous l'onglet Fichiers à gauche

# On peut le relire avec :

with open("test-pickle.pkl", "rb") as f: # rb = read + binaire
  valeurs_dans_le_fichier = pickle.load(f)

print("Le fichier contenait:", valeurs_dans_le_fichier)

Le fichier contenait: [1, 2, 3, 'ceci est un objet écrit dans un fichier']


On va se servir de ça pour charger un réseau de neurones pré-entraîné. Les coefficients de ce réseau sont ajustés pour fonctionner avec des sudokus

In [21]:
#@title Téléchargement et lecture d'un fichier pickle qui contient un modèle déjà entraîné
! (test -e mlp-sudoku-preentraine-sklearn.pkl && echo Fichier prêt) || wget https://raw.githubusercontent.com/316k/misc-data/refs/heads/main/mlp-sudoku-preentraine-sklearn.pkl -O mlp-sudoku-preentraine-sklearn.pkl

with open("mlp-sudoku-preentraine-sklearn.pkl", "rb") as f:
  modele_sudoku_preentraine = pickle.load(f)

print("=== Configuration ===")
print(modele_sudoku_preentraine)
print("- Couche d'entrée:", modele_sudoku_preentraine.coefs_[0].shape[0], "caractéristiques")
print("- Couches cachées:", modele_sudoku_preentraine.hidden_layer_sizes)
print("- Fonction d'activation des couches cachées:", modele_sudoku_preentraine.activation)
print("- Couche de sortie: 1 neurone, fonction d'activation finale:", modele_sudoku_preentraine.out_activation_)
print()

Fichier prêt
=== Configuration ===
MLPClassifier(hidden_layer_sizes=(1024, 512, 64))
- Couche d'entrée: 81 caractéristiques
- Couches cachées: (1024, 512, 64)
- Fonction d'activation des couches cachées: relu
- Couche de sortie: 1 neurone, fonction d'activation finale: logistic



/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelBinarizer from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MLPClassifier from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [15]:
#@title Évaluation du modèle pré-entraîné

# Nettoyage de la mémoire pour se laisser une chance de ne pas faire exploser la RAM
del X_entrainement, X_validation, X_test

X_complet = dataset[:, :-1]
y_complet = dataset[:, -1]
y_predictions_complet = modele_sudoku_preentraine.predict(X_complet)

print("Exactitude du modèle pré-entraîné sur le jeu de test:", accuracy_score(y_complet, y_predictions_complet))

Exactitude du modèle pré-entraîné sur le jeu de test: 1.0


Vous devriez voir une exactitude de 100%.

&nbsp;

Notez tout de même : ce 100% est **trompeur**. On est limités dans les tests qu'on fait, mais de mon côté, j'ai pu tester avec encore plus de grilles générées, et je trouve quelques rares cas d'échec pour mon modèle.

Avec des tests plus gros, je trouve un taux de 27 erreurs sur 2 000 000 (soit un taux de 99.99865% d'exactitude).

&nbsp;

Gardez bien en tête : **il existe une configuration de réseau de neurones qui marcherait à 100%**. On ne l'a cependant pas trouvée avec la descente de gradient.

## Boîte noire

Le réseau pré-entraîné a **$641~665$ paramètres**. On sait avec l'évaluation qu'il marche très, très bien, malgré quelques cas d'échecs rares.

Remarquez bien ici que les $641~665$ paramètres sont **très difficiles à interpréter**. On a un modèle de **boîte noire**, une machine qui marche sans qu'on soit facilement capables d'inspecter et de comprendre pourquoi elle marche.

**Si vous ne connaissiez pas déjà les règles d'un sudoku, le réseau de neurones appris ici ne vous aidera pas.**

&nbsp;

La régression linéaire et la classification linéaire, ça nous donne des coefficients interprétables facilement, mais ce n'est pas le cas des réseaux de neurones. Il est fort possible que certains neurones soient redondants, ou encore que certains des 1024 neurones de la première couche ne servent strictement à rien.

Il y a un peu de recherche qui se fait sur l'interprétabilité des réseaux, mais les réseaux de neurones ne sont pas fondamentalement des outils qui servent à comprendre un phénomène.

# En résumé

- La **mise à l'échelle des données** est essentielle pour diverses raisons dans plusieurs algorithmes d'apprentissage
- On devrait tester un algorithme simple sur notre problème avant de se lancer sur les réseaux de neurones
- Les réseaux de neurones sont très longs à entraîner
- La recherche d'hyperparamètres se fait souvent **au hasard** (ce n'est pas la seule stratégie, mais c'en est une simple qui contourne le problème de l'explosion combinatoire des hyperparamètres)
- Un réseau de neurones assez complexe peut **apprendre tout seul la non-linéarité de notre jeu de données**. Valider un sudoku n'est pas un problème linéaire (ie, qui se résoud avec une simple somme pondérée), et on atteint des résulats proches de 100% d'exactitude avec un réseau de neurone bien configuré
- L'entraînement n'a pas besoin d'être refait à toutes les fois, on peut **sauvegarder et recharger un modèle pré-entraîné** et l'utiliser directement les prochaines fois
- La solution parfaite avec 100% d'exactitude existe en théorie, mais la descente de gradient ne l'a pas trouvée
- Notre modèle final est une **boîte noire** : on ne comprend pas pourquoi les résultats sont bons, les ~640 000 paramètres sont très difficiles à interpréter